# Đào tạo mạng SOTA CTR-GCN cho Hệ Thống Báo Động (Transfer Learning)

Notebook này được thiết kế để chạy tự động quy trình chuẩn hóa. Tất cả code logic đã được tách ra thành file `.py` chuyên nghiệp.

In [ ]:
# 1. KẾT NỐI VỚI GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. TẢI MÃ NGUỒN TỪ GITHUB VÀ CÀI ĐẶT THƯ VIỆN
!if [ -d "realtime-abnormal-behavior-detection" ]; then cd realtime-abnormal-behavior-detection && git fetch origin && git reset --hard origin/dev; else git clone -b dev https://github.com/NBasLongz/realtime-abnormal-behavior-detection.git; fi
%cd /content/realtime-abnormal-behavior-detection
!pip install -r requirements.txt
!pip install onnx onnxscript onnxruntime

In [ ]:
# 3. CHÉP DỮ LIỆU TỪ DRIVE VÀO PROJECT
!cp -r /content/drive/MyDrive/Abnormal_Data/raw/* data/raw/

In [ ]:
# 4. TRÍCH XUẤT KHUNG XƯƠNG (Pose Estimation)
# Tự động bỏ qua các video đã trích xuất trước đó để tiết kiệm thời gian
!python scripts/data_processing/build_sequences.py

In [ ]:
# 5. CHUẨN HÓA TỌA ĐỘ VỀ TÂM CƠ THỂ [-1, 1], AUGMENTATION VÀ ĐỔI FORMAT CTR-GCN
!python scripts/data_processing/export_gcn_dataset.py

In [ ]:
# 6. HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH (GỒM BÁO CÁO F1, RECALL, CONFUSION MATRIX)
# Lưu output vào cả console và file text để bạn làm báo cáo
!python scripts/training/train_ctr_gcn.py | tee training_report.txt

In [ ]:
# 7. XUẤT ONNX VÀ SAO CHÉP MỌI KẾT QUẢ VỀ GOOGLE DRIVE
!python scripts/training/export_onnx.py

# a. Chép Model Weights (.onnx và .pt)
!cp weights/classification/ctrgcn_best.onnx /content/drive/MyDrive/Abnormal_Data/ 2>/dev/null || true
!cp weights/classification/ctrgcn_best.pt /content/drive/MyDrive/Abnormal_Data/ 2>/dev/null || true

# b. Chép Báo cáo đánh giá
!cp training_report.txt /content/drive/MyDrive/Abnormal_Data/ 2>/dev/null || true

# c. Nén toàn bộ dữ liệu khung xương đã chuẩn hóa
!zip -r processed_data.zip data/processed/
!cp processed_data.zip /content/drive/MyDrive/Abnormal_Data/ 2>/dev/null || true

print("TẤT CẢ KẾT QUẢ ĐÃ ĐƯỢC LƯU VỀ GOOGLE DRIVE CỦA BẠN!")